<a href="https://colab.research.google.com/github/Zainaansari/prompt-senstivity-in-propaganda-detection/blob/main/notebooks/run_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Sensitivity in Propaganda Detection - Colab Runner

Verifying the GPU is actually attached

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


Git clone to colab

In [29]:
import os

GITHUB_USERNAME = "Zainaansari"
REPO = "prompt-senstivity-in-propaganda-detection"

if not os.path.exists(REPO):
  !git clone https://github.com/{GITHUB_USERNAME}/{REPO}.git
  %cd {REPO}
else:
  !cd {REPO} && git pull origin

remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 11 (delta 2), reused 11 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 4.19 KiB | 858.00 KiB/s, done.
From https://github.com/Zainaansari/prompt-senstivity-in-propaganda-detection
   2db950e..56ec2c9  main       -> origin/main
Updating 2db950e..56ec2c9
Fast-forward
 config.yaml                     |  48 ++++++++++++++++++-
 notebooks/run_experiments.ipynb |  38 ++++++++++++++-
 pyproject.toml                  |  31 +++++++++++++
 setup.py                        |  14 ++++++
 src/model_runner.py             | 100 ++++++++++++++++++++++++++++++++++++++++
 src/prompt_builder.py           |  16 ++-----
 src/utils.py                    |  12 +++++
 7 files changed, 244 insertions(+), 15 deletions(-)
 create mode 100644 pyproject.toml
 create mode 100644 setup.py


In [27]:
!pip install -q -e .

ERROR: file:///content/prompt-senstivity-in-propaganda-detection does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


Installing hugging face

In [9]:
from huggingface_hub import login
login()

Load Qwen2.5-3B-Instruct in 4-bit and run one test generation

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
test_prompt = "What is the capital of France? Answer in one word."

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

What is the capital of France? Answer in one word. Paris.


In [13]:
import sys
sys.path.append('src')
from prompt_builder import build_prompt
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

labels = config["labels"]

# Use one real span for now — pull the first row from your processed data
import pandas as pd
df = pd.read_csv("data/processed/processed_data.csv")
print(df.columns.tolist())  # confirm the actual column names first
print(df.iloc[0])

['article_id', 'technique', 'start', 'end', 'text_spans']
article_id                                            111111111
technique                                   Appeal_to_Authority
start                                                       265
end                                                         323
text_spans    The next transmission could be more pronounced...
Name: 0, dtype: object


In [14]:
real_span = df.iloc[0]["text_spans"]
real_technique = df.iloc[0]["technique"]  # the gold label, for comparison later

prompt = build_prompt("standard", real_span, labels)
print(prompt)

Classify the following text span into exactly one of these propaganda techniques:
- Loaded Language
- Name Calling, Labeling
- Repetition
- Exaggeration, Minimization
- Doubt
- Appeal to Fear/Prejudice
- Flag-Waving
- Causal Oversimplification
- Slogans
- Appeal to Authority
- Black-and-White Fallacy
- Thought-terminating Cliches
- Bandwagon, Reductio ad Hitlerum
- Straw Men, Whataboutism, Red Herring

Text span: "The next transmission could be more pronounced or stronger"



In [15]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)
print("\n---")
print("Gold label was:", real_technique)

Classify the following text span into exactly one of these propaganda techniques:
- Loaded Language
- Name Calling, Labeling
- Repetition
- Exaggeration, Minimization
- Doubt
- Appeal to Fear/Prejudice
- Flag-Waving
- Causal Oversimplification
- Slogans
- Appeal to Authority
- Black-and-White Fallacy
- Thought-terminating Cliches
- Bandwagon, Reductio ad Hitlerum
- Straw Men, Whataboutism, Red Herring

Text span: "The next transmission could be more pronounced or stronger"
To classify the given text span "The next transmission could be more pronounced or stronger" using the provided propaganda techniques, we need to analyze the content and context of the statement.

1. **Loaded Language**: This technique involves using words that have strong emotional

---
Gold label was: Appeal_to_Authority


the model isn't following the "respond with only the technique name" instruction at all. Instead of a one-word answer, it's writing out an essay-style reasoning walkthrough, and got cut off mid-sentence at 50 tokens without ever reaching an answer.

In [17]:
messages = [
    {"role": "user", "content": prompt}
]

chat_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_input, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)

# Only decode the newly generated tokens, not the input echoed back
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)
print("\n---")
print("Gold label was:", real_technique)

The given text span "The next transmission could be more pronounced or stronger" does not directly employ any of the listed propaganda techniques. It is simply stating a factual possibility about the transmission's characteristics without using persuasive or manipulative language.

However, if we were to analyze it in terms of the closest technique, it might lean towards **Exaggeration, Minimization**. The phrase suggests that the transmission could be more pronounced or stronger, which implies a potential increase in intensity but doesn't necessarily exaggerate or minimize anything. Without additional context, it's hard to classify it precisely, but it doesn't fit neatly into the other categories either.

---
Gold label was: Appeal_to_Authority


In [18]:
prompt_structured = build_prompt("structured_output", real_span, labels)

messages = [{"role": "user", "content": prompt_structured}]
chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_input, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

{"technique": "Exaggeration, Minimization", "confidence": 0.9}


In [19]:
prompt_role = build_prompt("role_based", real_span, labels)

messages = [{"role": "user", "content": prompt_role}]
chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_input, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

The text span "The next transmission could be more pronounced or stronger" does not directly employ any of the specific propaganda techniques listed. However, it can be interpreted as potentially using **Exaggeration** if it implies that the next transmission will be significantly more intense than usual, which might be intended to create a sense of urgency or importance without providing concrete evidence.

If we strictly classify based on the provided list, this text span doesn't fit neatly into any single technique. It leans towards **Exaggeration**, but it's subtle and context-dependent. If you want to categorize it, I would lean towards **Exaggeration**.


In [22]:
article = """Next plague outbreak in Madagascar could be 'stronger': WHO

Geneva - The World Health Organisation chief on Wednesday said a deadly plague epidemic appeared to have been brought under control in Madagascar, but warned the next outbreak would likely be stronger.

"The next transmission could be more pronounced or stronger," WHO Director-General Tedros Adhanom Ghebreyesus told reporters in Geneva, insisting that "the issue is serious."

An outbreak of both bubonic plague, which is spread by infected rats via flea bites, and pneumonic plague, spread person to person, has killed more than 200 people in the Indian Ocean island nation since August.

Madagascar has suffered bubonic plague outbreaks almost every year since 1980, often caused by rats fleeing forest fires.

The disease tends to make a comeback each hot rainy season, from September to April.
On average, between 300 and 600 infections are recorded every year among a population approaching 25 million people, according to a UN estimate.

But Tedros voiced alarm that "plague in Madagascar behaved in a very, very different way this year."

Cases sprang up far earlier than usual and, instead of being confined to the countryside, the disease infiltrated towns.
The authorities recorded more than 2 000 cases, and Tedros said Wednesday the death toll stood at 207.

He also pointed to the presence of the pneumonic version, which spreads more easily and is more virulent, in the latest outbreak.

He praised the rapid response from WHO and Madagascar authorities that helped bring the outbreak under control, but warned that the danger was not over.

The larger-than-usual outbreak had helped spread the bacteria that causes the plague more widely.

This along with poor sanitation and vector control on Madagascar meant that "when (the plague) comes again it starts from more stock, and the magnitude in the next transmission could be higher than the one that we saw," Tedros said.

"That means that Madagascar could be affected more, and not only that, it could even spill over into neighbouring countries and beyond," he warned.

Complicating vector control is the fact that the fleas that carry the Yersinia pestis bacteria that causes the plague have proven to be widely resistant to chemicals and insecticides.

"That's a dangerous combination," Tedros said.
"""

In [24]:
article[265:323]

'The next transmission could be more pronounced or stronger'